In [0]:
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

def clean_orders(df: DataFrame) -> DataFrame:
    df = (df
          .filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull())
          .filter((F.col("quantity") > 0) & (F.col("unit_price") >= 0))
          .withColumn("status", F.upper(F.trim("status")))
          .withColumn("product", F.trim("product"))
          .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))
          .withColumn("updated_at", F.to_timestamp("updated_at")))
    w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def add_line_total(df: DataFrame) -> DataFrame:
    disc = F.coalesce(F.col("discount_pct"), F.lit(0.0))
    return df.withColumn(
        "line_total", F.round(F.col("quantity") * F.col("unit_price") * (1 - disc / 100), 2))

def join_customers(orders: DataFrame, customers: DataFrame) -> DataFrame:
    c = customers.dropDuplicates(["customer_id"])
    return (orders.join(c, "customer_id", "left")
            .withColumn("customer_name", F.coalesce("customer_name", F.lit("UNKNOWN")))
            .withColumn("country", F.coalesce("country", F.lit("UNKNOWN"))))

def monthly_revenue(df: DataFrame) -> DataFrame:
    return (df.filter(F.col("status") != "CANCELLED")
            .withColumn("order_month", F.date_format("order_date", "yyyy-MM"))
            .groupBy("order_month", "country")
            .agg(F.round(F.sum("line_total"), 2).alias("revenue"),
                 F.count("order_id").alias("order_count")))